# RV recovery

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [GitHub](https://github.com/oirlab/HISPEC_DRP).</span>

Recover an injected radial velocity from the rest-frame **BT-Settl lte040**
template on three paths:

1. **Native L2** — paint the shifted spectrum onto the echelle WAVE grid.
2. **Rate map → Stages 1–2** — paint a rate map, extract, and blaze.
3. **UTR → Stages 0–2** — same spectrum through a noiseless ramp and Stage 0.

Before the recovery grid, the notebook builds a simulated LFC wavelength
solution and an identity fiber flat so every path uses the same calibrations.


In [ ]:
from pathlib import Path

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

OUT = Path("/Users/ckilpatrick/rv_recovery")
OUT.mkdir(parents=True, exist_ok=True)
print("output:", OUT.resolve())


hispecdrp 0.0.post957+gf8dfcb577.d20260828 @ /Users/ckilpatrick/Dropbox/scripts/python/cursor/HISPEC_DRP/hispecdrp/__init__.py


OSError: [Errno 30] Read-only file system: '/path'

## 1. Template, mask, and truth spectra

Load the densified rest-frame template, Doppler-shift it onto the recovery
grid, and confirm zero-RV recovery with the packaged order-136 fine-paint
ccfopt mask (`*.order136.iso14.nfine512.ccfopt.delta.mas`). Regenerate with
`python hispecdrp/data/masks/regenerate_order136_nfine_ccfopt_mask.py`.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import (
    STEP_BULK_RV_KM_S,
    STELLAR_RV_NO_REJECT_KW,
    doppler_shift_spectrum,
    load_densified_template,
    run_step_grid,
    verify_zero_rv_recovery,
)
from hispecdrp.utils.line_mask_utils import (
    package_data_masks_dir,
    read_mask_generation_meta,
)

FIBER = 1
ORDER = 136
TEMPLATE = "lte040.0-4.5-0.0a+0.0.BT-Settl.spec.7.h5"
# Pregenerated fine-paint (N_FINE=512) ccfopt rezero for order 136 iso14.
MASK_PATH = (
    package_data_masks_dir()
    / "lte040.0-4.5-0.0a+0.0.BT-Settl.spec.7.order136.iso14.nfine512.ccfopt.delta.mas"
)
if not MASK_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {MASK_PATH}; run:\n"
        "  python hispecdrp/data/masks/regenerate_order136_nfine_ccfopt_mask.py"
    )
MASK_PATH = str(MASK_PATH.resolve())
BULK_RV_KM_S = STEP_BULK_RV_KM_S.copy()
meta = read_mask_generation_meta(MASK_PATH) or {}
print(f"mask: {MASK_PATH}")
print(
    f"n_lines={meta.get('n_lines')}  order={meta.get('order', ORDER)}  "
    f"n_fine={meta.get('n_fine')}  "
    f"outlier_sigma={STELLAR_RV_NO_REJECT_KW['outlier_sigma']:.0e}"
)
print(f"grid={list(BULK_RV_KM_S)} km/s")

wl_rest, flux_rest = load_densified_template(TEMPLATE)
print(f"truth 1D: n={wl_rest.size}  lambda={wl_rest[0]:.4f}-{wl_rest[-1]:.4f}")

verify_zero_rv_recovery(
    template=TEMPLATE,
    mask_path=str(MASK_PATH),
    fiber=FIBER,
)

TRUTH = {
    float(rv): doppler_shift_spectrum(wl_rest, flux_rest, float(rv))
    for rv in BULK_RV_KM_S
}
summaries = {}


## 2. Simulated LFC wavelength solution

Paint an LFC rate map and fit a wavelength solution (and empirical LSF) with
the same ProcessLFC settings used in the simulated LFC example. Products go
under `OUT/lfc_wavesol_cals/`.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import derive_lfc_wavesol

lfc_wavesol, lfc_info = derive_lfc_wavesol(
    spectrograph="BSPEC",
    fiber=FIBER,
    output_dir=OUT / "lfc_wavesol_cals",
    derive_lsf=True,
)
lfc_lsf = lfc_info["lsf"]
print(lfc_info)
print(
    f"LFC WAVE shape={getattr(lfc_wavesol, f'wave_fib{FIBER}').shape}  "
    f"path={lfc_info['wavesol_path']}"
)
print(f"LFC LSF path={lfc_info['lsf_path']}")
assert lfc_lsf is not None


## 3. Identity fiber flat

Build Stage-1 masters once and replace the continuum fiber flat with ones
shaped like the LFC WAVE, so blaze correction does not reweight the CCF.


In [ ]:
import numpy as np

from hispecdrp.simulations.ideal_spectrum_rv import (
    FIBERFLAT_IDENTITY,
    prepare_stage1_cals,
)
from hispecdrp.simulations.simulate_rate_map import SimulateRateMap
from hispecdrp.simulations.spectra import quiet_pipeline_logging

FIBERFLAT_MODE = FIBERFLAT_IDENTITY

_seed_rv = 0.0 if 0.0 in TRUTH else float(next(iter(TRUTH)))
with quiet_pipeline_logging():
    srm = SimulateRateMap()
    srm.set_fiber_spectrum(
        fiber_number=FIBER,
        spectrum_wavelengths=TRUTH[_seed_rv][0],
        spectrum_flux=TRUTH[_seed_rv][1],
        continuous=False,
    )
    rate0 = np.asarray(
        srm.create_native_rate_map(spectrograph="BSPEC", save=False),
        dtype=np.float64,
    )
    stage1_cals = prepare_stage1_cals(
        OUT,
        rate0,
        spectrograph="BSPEC",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    )

ff = stage1_cals["fiberflat"]
ff_spec = np.asarray(getattr(ff, f"spec_fib{FIBER}"))
print(f"fiberflat_source={stage1_cals['fiberflat_source']!r}  mode={FIBERFLAT_MODE!r}")
print(f"fiberflat shape={ff_spec.shape}  unique={np.unique(ff_spec)}")
print(f"cached under: {OUT / 'cals'}")
assert stage1_cals["fiberflat_source"] == "identity_ones"
assert np.allclose(ff_spec, 1.0)


## 4. Native L2 injection

Paint each Doppler-shifted truth spectrum onto a native BSPEC L2 and recover RV.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_l2


def recover_l2(bulk):
    wl, flux = TRUTH[float(bulk)]
    return recover_rv_from_truth_l2(
        bulk,
        wavelengths=wl,
        flux=flux,
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        fiber=FIBER,
    )


summaries["L2"] = run_step_grid(
    recover_l2,
    name="L2",
    title="BT-Settl lte040 (truth 1D -> L2)",
    out_path=OUT / "lte040_step_l2.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 5. Rate map → Stages 1–2

Paint a native rate map, box-extract, apply the LFC WAVE, and blaze with the
identity fiber flat before recovering RV.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_rate_map

summaries["rate_map"] = run_step_grid(
    lambda rv: recover_rv_from_truth_rate_map(
        rv,
        wavelengths=TRUTH[float(rv)][0],
        flux=TRUTH[float(rv)][1],
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        output_dir=OUT / "rate_map",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    ),
    name="rate_map",
    title=f"Rate map -> Stages 1-2 (LFC WAVE + {FIBERFLAT_MODE} FF)",
    out_path=OUT / "rate_map_recovery.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 6. UTR → Stages 0–2

Same calibrations after a noiseless up-the-ramp exposure and Stage 0.


In [ ]:
from hispecdrp.simulations.rv_step_diagnostics import recover_rv_from_truth_ramp

summaries["ramp"] = run_step_grid(
    lambda rv: recover_rv_from_truth_ramp(
        rv,
        wavelengths=TRUTH[float(rv)][0],
        flux=TRUTH[float(rv)][1],
        template=TEMPLATE,
        mask_path=str(MASK_PATH),
        output_dir=OUT / "ramp",
        fiber=FIBER,
        wavesol=lfc_wavesol,
        fiberflat_mode=FIBERFLAT_MODE,
    ),
    name="ramp",
    title=f"UTR -> Stages 0-2 (LFC WAVE + {FIBERFLAT_MODE} FF)",
    out_path=OUT / "ramp_recovery.png",
    bulk_rv_km_s=BULK_RV_KM_S,
)


## 7. Compare paths

Summarize mean bias and scatter across the three recovery paths.


In [ ]:
from hispecdrp.simulations.plotting import plot_rv_step_comparison
from hispecdrp.simulations.rv_step_diagnostics import compare_step_summaries

order = ["L2", "rate_map", "ramp"]
present = [k for k in order if k in summaries]
compare_step_summaries({k: summaries[k] for k in present})
plot_rv_step_comparison(
    {k: summaries[k] for k in present},
    out_path=OUT / "rv_recovery_comparison.png",
)
